# 01 · Exploratory data analysis
Amazon Reviews 2023, **Video_Games 5-core**, official absolute-timestamp split.

Sections 1-4 follow the proposal's *Data Preparation* bullets. Run `uv run vgrec download && uv run vgrec prepare` first.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from vgrec.analysis import history_bucket
from vgrec.config import T1_MS, T2_MS
from vgrec.data import load_dataset, load_items, load_raw_interactions, quality_report
from vgrec.protocol import make_split

ds = load_dataset()
items = load_items()
inter = ds.interactions
splits = {stage: make_split(ds, stage) for stage in ('valid', 'test')}
density = len(inter) / (ds.n_users * ds.n_items)
print(f'{ds.n_users:,} users · {ds.n_items:,} items · {len(inter):,} interactions · density {density:.2e}')

## 1. Data quality: missing values, invalid ratings, duplicates, split boundaries

In [ ]:
pd.Series(quality_report(load_raw_interactions()), name='count').to_frame()

## 2. Sparsity and distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
inter['rating'].value_counts().sort_index().plot.bar(ax=axes[0], title='Rating distribution')
for ax, col, label in [(axes[1], 'user_idx', 'user'), (axes[2], 'item_idx', 'item')]:
    counts = inter[col].value_counts().to_numpy()
    ax.loglog(np.arange(1, len(counts) + 1), counts)
    ax.set(title=f'Interactions per {label} (rank-ordered)', xlabel=f'{label} rank', ylabel='interactions')
plt.tight_layout()

In [ ]:
pd.DataFrame({
    'per user': inter.groupby('user_idx').size().describe(),
    'per item': inter.groupby('item_idx').size().describe(),
}).round(1)

## 3. Time: split boundaries and popularity drift

In [ ]:
monthly = inter.set_index(pd.to_datetime(inter['timestamp'], unit='ms')).resample('MS').size()
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(monthly.index, monthly.to_numpy())
for t, name in ((T1_MS, 'T1 (valid starts)'), (T2_MS, 'T2 (test starts)')):
    ax.axvline(pd.to_datetime(t, unit='ms'), color='red', ls='--', label=name)
ax.set(title='Interactions per month', ylabel='interactions')
ax.legend();

In [ ]:
def top_titles(split_names, n=10):
    top = inter.loc[inter['split'].isin(split_names), 'item_idx'].value_counts().head(n).index
    return items['title'].iloc[top].str.slice(0, 60).tolist()

pd.DataFrame({'train (all history)': top_titles(['train']),
              'valid period': top_titles(['valid']),
              'test period': top_titles(['test'])})

## 4. Cold start under the timestamp split
Rows whose user has no history, or whose item never appears in the history, cannot be ranked by any model and are dropped from the targets.

In [ ]:
pd.DataFrame({stage: split.stats for stage, split in splits.items()})

In [ ]:
pd.DataFrame({stage: pd.Series(history_bucket(split.history_lengths)).value_counts(sort=False)
              for stage, split in splits.items()})

## 5. Item metadata completeness and text

In [ ]:
fields = ['title', 'store', 'categories', 'features', 'description', 'details']
completeness = (items[fields] != '').mean().rename('share non-empty').to_frame()
words = items['item_text'].str.split().str.len().describe().rename('words per item_text')
display(completeness.round(3))
words.round(0).to_frame()

## Observations
Write 4-6 bullet points for the report: sparsity, rating skew, temporal drift, cold-start shares, metadata gaps, and what each implies for the models.